In [1]:
import pandas as pd

df = pd.read_parquet("hf://datasets/google-research-datasets/go_emotions/raw/train-00000-of-00001.parquet")

/opt/anaconda3/envs/ml/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df.columns

Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='str')

In [3]:
non_emotion_cols = [
    'text', 'id', 'author', 'subreddit', 'link_id',
    'parent_id', 'created_utc', 'rater_id', 'example_very_unclear', 'neutral'
]

emotion_cols = [col for col in df.columns if col not in non_emotion_cols]
print(emotion_cols)

['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise']


Quick Checks

In [4]:
print("Shape:", df.shape)
print("Number of emotion columns:", len(emotion_cols))
print(df[emotion_cols].sum().sort_values(ascending=False).head(10))
print(df['example_very_unclear'].value_counts(dropna=False))

Shape: (211225, 37)
Number of emotion columns: 27
approval          17620
admiration        17131
annoyance         13618
gratitude         11625
disapproval       11424
curiosity          9692
amusement          9245
realization        8785
optimism           8715
disappointment     8469
dtype: int64
example_very_unclear
False    211225
Name: count, dtype: int64


### Aggregating to column level

`example_very_unclear`

In [5]:
agg_dict = {col: 'max' for col in emotion_cols}
agg_dict['example_very_unclear'] = 'max'

df_clean = (
    df.groupby('id', as_index=False)
      .agg(agg_dict)
      .rename(columns={'id': 'comment_id'})
)

Computing multi-label-count

In [6]:
df_clean['multi_label_count'] = df_clean[emotion_cols].sum(axis=1).astype(int)

`unclear_fraction`

In [7]:
unclear_fraction = (
    df.groupby('id')['example_very_unclear']
      .mean()
      .reset_index()
      .rename(columns={'id': 'comment_id', 'example_very_unclear': 'unclear_fraction'})
)

# merge
df_clean = df_clean.merge(unclear_fraction, on='comment_id')


### Final Feature Dataset

In [8]:
feature_df = df_clean[
    ['comment_id', 'multi_label_count', 'example_very_unclear', 'unclear_fraction']
].copy()

feature_df.to_csv("comment_multilabel_unclear.csv", index=False)

### Quick Checks

In [9]:
print("\nShape:", feature_df.shape)
print(feature_df.head())

print("\nUnique IDs check:", feature_df['comment_id'].nunique(), len(feature_df))

print("\nMulti-label stats:")
print(feature_df['multi_label_count'].describe())

print("\nUnclear (max):")
print(feature_df['example_very_unclear'].value_counts(dropna=False))

print("\nUnclear fraction:")
print(feature_df['unclear_fraction'].describe())


Shape: (58011, 4)
  comment_id  multi_label_count  example_very_unclear  unclear_fraction
0    eczazk6                  1                 False               0.0
1    eczb07q                  0                 False               0.0
2    eczb4bm                  1                 False               0.0
3    eczb527                  2                 False               0.0
4    eczb6r7                  4                 False               0.0

Unique IDs check: 58011 58011

Multi-label stats:
count    58011.000000
mean         2.306925
std          1.473224
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         13.000000
Name: multi_label_count, dtype: float64

Unclear (max):
example_very_unclear
False    58011
Name: count, dtype: int64

Unclear fraction:
count    58011.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: unclear_fraction, dtype: float64
